In [0]:
# Smart Data Deduplication Pipeline
# Notebook 4: Duplicate Detection

DATABASE_NAME = "smart_data_deduplication"

print("Using database:", DATABASE_NAME)
print("Starting duplicate detection...")

Using database: smart_data_deduplication
Starting duplicate detection...


In [0]:
# Load validated records from Delta table

valid_df = spark.table(
    f"{DATABASE_NAME}.validated_records"
)

print(
    "Valid records available for duplicate detection:",
    valid_df.count()
)

display(valid_df)

Valid records available for duplicate detection: 54


record_id,entity_type,name,category,description,value,source,updated_at,test_case
P0001,product,Apple iPhone 15,Electronics,Smartphone 128GB,69999.0,source_B,2026-09-01 09:01:00,POSITIVE_CLEAN
P0002,product,Samsung Galaxy S24,Electronics,Smartphone 256GB,74999.0,source_C,2026-09-01 09:02:00,POSITIVE_CLEAN
P0003,product,Dell Inspiron 15,Computers,Laptop 15 inch,58999.0,source_D,2026-09-01 09:03:00,POSITIVE_CLEAN
P0004,product,Sony WH-1000XM5,Electronics,Wireless headphones,29999.0,source_E,2026-09-01 09:04:00,POSITIVE_CLEAN
P0005,product,Logitech MX Master 3S,Accessories,Wireless mouse,8499.0,source_F,2026-09-01 09:05:00,POSITIVE_CLEAN
P0006,product,HP Pavilion 14,Computers,Laptop 14 inch,65999.0,source_G,2026-09-01 09:06:00,POSITIVE_CLEAN
P0007,product,Canon EOS R10,Cameras,Mirrorless camera,82999.0,source_A,2026-09-01 09:07:00,POSITIVE_CLEAN
P0008,product,OnePlus 12,Electronics,Smartphone 256GB,64999.0,source_B,2026-09-01 09:08:00,POSITIVE_CLEAN
B0009,book,The Alchemist,Books,Novel by Paulo Coelho,399.0,source_C,2026-09-01 09:09:00,POSITIVE_CLEAN
B0010,book,Atomic Habits,Books,Self improvement book,499.0,source_D,2026-09-01 09:10:00,POSITIVE_CLEAN


In [0]:
# Exact duplicate detection

from pyspark.sql.functions import col

left = valid_df.alias("left")
right = valid_df.alias("right")

exact_duplicates = (
    left.join(
        right,
        (
            (col("left.record_id") < col("right.record_id")) &
            (col("left.name") == col("right.name")) &
            (col("left.category") == col("right.category")) &
            (col("left.description") == col("right.description")) &
            (col("left.value") == col("right.value"))
        ),
        "inner"
    )
    .select(
        col("left.record_id").alias("record_id"),
        col("right.record_id").alias("duplicate_record_id"),
        col("left.name").alias("name"),
        col("left.category").alias("category"),
        col("left.value").alias("value")
    )
)

print("Exact duplicate pairs:", exact_duplicates.count())

display(exact_duplicates)

Exact duplicate pairs: 0


record_id,duplicate_record_id,name,category,value


In [0]:
# Improved Near-Duplicate Detection
# Levenshtein Distance with stronger similarity threshold

from pyspark.sql.functions import (
    col,
    levenshtein,
    greatest,
    length,
    round,
    when
)

left = valid_df.alias("left")
right = valid_df.alias("right")

near_duplicates = (
    left.join(
        right,
        col("left.record_id") < col("right.record_id"),
        "inner"
    )
    .withColumn(
        "edit_distance",
        levenshtein(
            col("left.name"),
            col("right.name")
        )
    )
    .withColumn(
        "max_name_length",
        greatest(
            length(col("left.name")),
            length(col("right.name"))
        )
    )
    .withColumn(
        "similarity_score",
        round(
            when(
                col("max_name_length") == 0,
                1.0
            ).otherwise(
                1.0 -
                (
                    col("edit_distance") /
                    col("max_name_length")
                )
            ),
            4
        )
    )
    .filter(
        (col("similarity_score") >= 0.93) &
        (col("similarity_score") < 1.0)
    )
    .select(
        col("left.record_id").alias("record_id_1"),
        col("right.record_id").alias("record_id_2"),
        col("left.name").alias("name_1"),
        col("right.name").alias("name_2"),
        col("left.category").alias("category_1"),
        col("right.category").alias("category_2"),
        col("left.test_case").alias("test_case_1"),
        col("right.test_case").alias("test_case_2"),
        col("edit_distance"),
        col("similarity_score")
    )
    .orderBy(col("similarity_score").desc())
)

print("Near-duplicate candidates:", near_duplicates.count())

display(near_duplicates)

Near-duplicate candidates: 2


record_id_1,record_id_2,name_1,name_2,category_1,category_2,test_case_1,test_case_2,edit_distance,similarity_score
P0002,P0006,Samsung Galaxy S24,Samsng Galaxy S24,Electronics,Computers,NEGATIVE_EXACT_DUPLICATE,NEGATIVE_NEAR_DUPLICATE,1,0.9444
P0002,P0006,Samsung Galaxy S24,Samsng Galaxy S24,Electronics,Computers,POSITIVE_CLEAN,NEGATIVE_NEAR_DUPLICATE,1,0.9444


In [0]:
# Combine exact and near-duplicate results

from pyspark.sql.functions import lit

# Format exact duplicates
exact_result = (
    exact_duplicates
    .select(
        col("record_id").alias("record_id_1"),
        col("duplicate_record_id").alias("record_id_2"),
        col("name").alias("name_1"),
        col("name").alias("name_2"),
        col("category").alias("category_1"),
        col("category").alias("category_2"),
        lit("EXACT_DUPLICATE").alias("duplicate_type"),
        lit(1.0).alias("similarity_score")
    )
)

# Format near duplicates
near_result = (
    near_duplicates
    .select(
        "record_id_1",
        "record_id_2",
        "name_1",
        "name_2",
        "category_1",
        "category_2",
        lit("NEAR_DUPLICATE").alias("duplicate_type"),
        "similarity_score"
    )
)

# Combine both
all_duplicates = (
    exact_result
    .unionByName(near_result)
    .orderBy(col("similarity_score").desc())
)

print("Total duplicate pairs:", all_duplicates.count())

display(all_duplicates)

Total duplicate pairs: 2


record_id_1,record_id_2,name_1,name_2,category_1,category_2,duplicate_type,similarity_score
P0002,P0006,Samsung Galaxy S24,Samsng Galaxy S24,Electronics,Computers,NEAR_DUPLICATE,0.9444
P0002,P0006,Samsung Galaxy S24,Samsng Galaxy S24,Electronics,Computers,NEAR_DUPLICATE,0.9444


In [0]:
# Save duplicate detection results to Delta

all_duplicates.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{DATABASE_NAME}.duplicate_detection_results")

print("Duplicate detection results saved successfully!")
print(
    "Total duplicate pairs:",
    spark.table(
        f"{DATABASE_NAME}.duplicate_detection_results"
    ).count()
)

display(
    spark.table(
        f"{DATABASE_NAME}.duplicate_detection_results"
    )
)

Duplicate detection results saved successfully!
Total duplicate pairs: 2


record_id_1,record_id_2,name_1,name_2,category_1,category_2,duplicate_type,similarity_score
P0002,P0006,Samsung Galaxy S24,Samsng Galaxy S24,Electronics,Computers,NEAR_DUPLICATE,0.9444
P0002,P0006,Samsung Galaxy S24,Samsng Galaxy S24,Electronics,Computers,NEAR_DUPLICATE,0.9444


In [0]:
from pyspark.sql import functions as F
import builtins

# Total records evaluated
total_records = valid_df.count()

# Exact duplicate records
exact_count = exact_duplicates.select("duplicate_record_id").distinct().count()

# Near-duplicate records
near_count = near_duplicates.select("record_id_2").distinct().count()

# Combine duplicate records without double-counting
all_duplicate_ids = (
    exact_duplicates.select(F.col("duplicate_record_id").alias("row_id"))
    .unionByName(
        near_duplicates.select(F.col("record_id_2").alias("row_id"))
    )
    .distinct()
)

total_duplicates = all_duplicate_ids.count()

# Non-duplicate records
non_duplicate_count = total_records - total_duplicates

# Calculate percentage using Python's built-in round
def calculate_rate(count):
    if total_records == 0:
        return 0.0
    return builtins.round((count / total_records) * 100, 2)

# Prepare Table 4
table4_data = [
    ("Exact duplicates", exact_count, calculate_rate(exact_count)),
    ("Near-duplicates", near_count, calculate_rate(near_count)),
    ("Total duplicates", total_duplicates, calculate_rate(total_duplicates)),
    ("Non-duplicate records", non_duplicate_count,
     calculate_rate(non_duplicate_count))
]

# Create Spark DataFrame
table4_df = spark.createDataFrame(
    table4_data,
    ["duplicate_type", "records_detected", "detection_rate"]
)

# Format percentage
table4_df = table4_df.withColumn(
    "detection_rate",
    F.concat(
        F.format_number(F.col("detection_rate"), 2),
        F.lit("%")
    )
)

# Display Table 4
print("TABLE 4: DUPLICATE DETECTION RESULTS")
print("Total records evaluated:", total_records)

display(table4_df)

# Save summary
table4_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"{DATABASE_NAME}.duplicate_detection_summary"
    )

print("Table 4 saved successfully!")

TABLE 4: DUPLICATE DETECTION RESULTS
Total records evaluated: 54


duplicate_type,records_detected,detection_rate
Exact duplicates,0,0.00%
Near-duplicates,1,1.85%
Total duplicates,1,1.85%
Non-duplicate records,53,98.15%


Table 4 saved successfully!
